<a href="https://colab.research.google.com/github/Mr-Kondo/_Inbox/blob/main/test_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Google Colab RAGシステム - 完全版
# 性能比較機能付き

## 実行環境
- Python 3.10+ (Google Colab標準環境)

## 必要なライブラリ
以下のライブラリが必要です。インストールは次のコードセルで行われます。
- docling
- sentence-transformers
- chromadb
- transformers
- accelerate
- bitsandbytes
- torch
- pypdf
- pandas
- plotly
- langchain
- langchain-community
- rank-bm25

In [3]:
# Google Colab RAGシステム - 完全版
# 性能比較機能付き

!python -m pip install -q docling sentence-transformers chromadb transformers accelerate bitsandbytes \
    torch pypdf pandas plotly langchain langchain-community rank-bm25

import os
import time
import json
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, asdict
import numpy as np

# Docling関連
from docling.document_converter import DocumentConverter

# Embedding & Vector DB
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

# LLM
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Reranker
from sentence_transformers import CrossEncoder

# Chunking
from langchain.text_splitter import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter
)

print("✅ ライブラリのインポート完了")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 9.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 6.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.3/245.3 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.7/20.7 MB 83.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.6/323.6 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 106.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.5/164.5 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━

In [13]:
# ================================================================================
# 設定クラス
# ================================================================================

@dataclass
class ChunkingConfig:
    """チャンキング設定"""
    method: str  # "recursive" or "semantic"
    chunk_size: int
    chunk_overlap: int
    overlap_ratio: float  # 重複率（0.0-1.0）

    def __post_init__(self):
        # overlap_ratioからchunk_overlapを計算
        if self.overlap_ratio > 0:
            self.chunk_overlap = int(self.chunk_size * self.overlap_ratio)

@dataclass
class ExperimentResult:
    """実験結果"""
    config: RAGConfig
    query: str
    answer: str
    execution_time: float
    retrieved_chunks: List[str]
    scores: List[float]
    search_time: float = 0.0  # 検索時間
    rerank_time: float = 0.0 # Rerank時間
    llm_time: float = 0.0     # LLM生成時間

In [5]:
# ================================================================================
# PDFドキュメント処理
# ================================================================================

class DocumentProcessor:
    """Doclingを使用したPDF処理"""

    def __init__(self):
        self.converter = DocumentConverter()

    def process_pdf(self, pdf_path: str) -> str:
        """PDFをテキストに変換"""
        print(f"📄 PDF処理開始: {pdf_path}")
        result = self.converter.convert(pdf_path)
        text = result.document.export_to_markdown()
        print(f"✅ 抽出完了: {len(text)}文字")
        return text

In [6]:
# ================================================================================
# チャンキング
# ================================================================================

class DocumentChunker:
    """複数のチャンキング手法を提供"""

    @staticmethod
    def chunk_recursive(text: str, config: ChunkingConfig) -> List[str]:
        """再帰的文字分割"""
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=config.chunk_size,
            chunk_overlap=config.chunk_overlap,
            separators=["\n\n", "\n", "。", "、", " ", ""]
        )
        chunks = splitter.split_text(text)
        print(f"🔪 再帰的チャンキング: {len(chunks)}チャンク生成")
        return chunks

    @staticmethod
    def chunk_semantic(text: str, config: ChunkingConfig,
                       embedder: SentenceTransformer) -> List[str]:
        """セマンティックチャンキング（意味的な区切り）"""
        # まず文単位で分割
        sentences = text.replace("。", "。\n").split("\n")
        sentences = [s.strip() for s in sentences if s.strip()]

        if len(sentences) <= 1:
            return sentences

        # 文埋め込みを計算
        embeddings = embedder.encode(sentences, show_progress_bar=False)

        # 隣接文間のコサイン類似度を計算
        similarities = []
        for i in range(len(embeddings) - 1):
            sim = np.dot(embeddings[i], embeddings[i+1]) / (
                np.linalg.norm(embeddings[i]) * np.linalg.norm(embeddings[i+1])
            )
            similarities.append(sim)

        # 類似度の低い箇所で分割（意味的な境界）
        threshold = np.percentile(similarities, 30)  # 下位30%を境界とする

        chunks = []
        current_chunk = sentences[0]
        current_length = len(sentences[0])

        for i, sentence in enumerate(sentences[1:], 1):
            # 意味的境界かつサイズ制約
            if (similarities[i-1] < threshold and
                current_length > config.chunk_size * 0.5):
                chunks.append(current_chunk)
                current_chunk = sentence
                current_length = len(sentence)
            else:
                current_chunk += sentence
                current_length += len(sentence)

                # 最大サイズに達したら強制分割
                if current_length > config.chunk_size:
                    chunks.append(current_chunk)
                    current_chunk = ""
                    current_length = 0

        if current_chunk:
            chunks.append(current_chunk)

        print(f"🧠 セマンティックチャンキング: {len(chunks)}チャンク生成")
        return chunks

    @classmethod
    def chunk(cls, text: str, config: ChunkingConfig,
              embedder: Optional[SentenceTransformer] = None) -> List[str]:
        """設定に応じてチャンキング"""
        if config.method == "semantic":
            if embedder is None:
                raise ValueError("セマンティックチャンキングにはembedderが必要です")
            return cls.chunk_semantic(text, config, embedder)
        else:
            return cls.chunk_recursive(text, config)

In [7]:
# ================================================================================
# ベクトルデータベース
# ================================================================================

class VectorDatabase:
    """ChromaDBを使用したベクトルストア"""

    def __init__(self, collection_name: str = "rag_collection"):
        self.client = chromadb.Client(Settings(anonymized_telemetry=False))
        self.collection_name = collection_name
        # 使用可能な公開モデルに変更
        self.embedder = SentenceTransformer('intfloat/multilingual-e5-large')
        print(f"🔮 intfloat/multilingual-e5-large 埋め込みモデル読み込み完了")

        # コレクション初期化
        try:
            self.client.delete_collection(name=collection_name)
        except:
            pass
        self.collection = self.client.create_collection(name=collection_name)

    def add_documents(self, chunks: List[str]):
        """ドキュメントをベクトルDBに追加"""
        print(f"💾 {len(chunks)}チャンクをベクトル化中...")

        embeddings = self.embedder.encode(
            chunks,
            show_progress_bar=True,
            batch_size=32
        )

        self.collection.add(
            embeddings=embeddings.tolist(),
            documents=chunks,
            ids=[f"chunk_{i}" for i in range(len(chunks))]
        )
        print("✅ ベクトルDB構築完了")

    def search(self, query: str, top_k: int = 5) -> Tuple[List[str], List[float]]:
        """クエリに基づいて検索"""
        query_embedding = self.embedder.encode([query])[0]

        results = self.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k
        )

        documents = results['documents'][0]
        distances = results['distances'][0]

        # 距離をスコアに変換（距離が小さいほどスコアが高い）
        scores = [1 / (1 + d) for d in distances]

        return documents, scores

In [8]:
# ================================================================================
# Reranker
# ================================================================================

class Reranker:
    """日本語対応Reranker"""

    def __init__(self):
        # 日本語対応のクロスエンコーダーモデル
        self.model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
        print("🎯 Rerankerモデル読み込み完了")

    def rerank(self, query: str, documents: List[str],
               top_k: int = 3) -> Tuple[List[str], List[float]]:
        """ドキュメントを再ランキング"""
        pairs = [[query, doc] for doc in documents]
        scores = self.model.predict(pairs)

        # スコアでソート
        ranked_indices = np.argsort(scores)[::-1][:top_k]

        reranked_docs = [documents[i] for i in ranked_indices]
        reranked_scores = [float(scores[i]) for i in ranked_indices]

        return reranked_docs, reranked_scores

In [9]:
# ================================================================================
# LLM生成
# ================================================================================

class JapaneseLLM:
    """日本語特化LLM（ELYZA）"""

    def __init__(self, model_name: str = "elyza/ELYZA-japanese-Llama-2-7b"):
        print(f"🤖 LLM読み込み中: {model_name}")

        # 4bit量子化設定
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            # メモリ不足対策としてオフロードを有効化
            llm_int8_enable_fp32_cpu_offload=True,
        )

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=quantization_config,
            # メモリ不足対策としてdevice_mapをautoに設定
            device_map="auto",
            trust_remote_code=True
        )
        print("✅ LLM読み込み完了")

    def generate(self, prompt: str, max_new_tokens: int = 512) -> str:
        """テキスト生成"""
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.7,
                do_sample=True,
                top_p=0.9,
                pad_token_id=self.tokenizer.eos_token_id
            )

        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        # プロンプト部分を除去
        response = response.split("回答：")[-1].strip()
        return response

In [10]:
# ================================================================================
# RAGシステム
# ================================================================================

class RAGSystem:
    """統合RAGシステム"""

    def __init__(self, config: RAGConfig, llm: JapaneseLLM, chroma_client: chromadb.Client, reranker: Optional[Reranker] = None, collection_name: str = "rag_collection"):
        self.config = config
        self.llm = llm # 外部から渡されたLLMインスタンスを使用
        self.reranker = reranker # 外部から渡されたRerankerインスタンスを使用

        # VectorDatabaseはExperimentManagerから渡されたクライアントを使用し、コレクションを初期化
        self.client = chroma_client
        self.collection_name = collection_name
        # 使用可能な公開モデルに変更 (埋め込みモデルはVectorDatabase内で初期化)
        self.embedder = SentenceTransformer('intfloat/multilingual-e5-large')
        print(f"🔮 intfloat/multilingual-e5-large 埋め込みモデル読み込み完了")

        # コレクション初期化 (実験ごとに新しいコレクションを作成)
        try:
            self.client.delete_collection(name=collection_name)
        except:
            pass
        self.collection = self.client.create_collection(name=collection_name)


    def index_document(self, pdf_path: str):
        """ドキュメントをインデックス化"""
        # PDF処理
        processor = DocumentProcessor()
        text = processor.process_pdf(pdf_path)

        # チャンキング
        chunks = DocumentChunker.chunk(
            text,
            self.config.chunking,
            embedder=self.embedder # RAGSystem内で初期化されたembedderを使用
        )

        # ベクトルDB構築
        self.add_documents(chunks) # self.vector_db.add_documents から変更

        return len(chunks)

    def add_documents(self, chunks: List[str]):
        """ドキュメントをベクトルDBに追加"""
        print(f"💾 {len(chunks)}チャンクをベクトル化中...")

        embeddings = self.embedder.encode(
            chunks,
            show_progress_bar=True,
            batch_size=32
        )

        self.collection.add(
            embeddings=embeddings.tolist(),
            documents=chunks,
            ids=[f"chunk_{i}" for i in range(len(chunks))]
        )
        print("✅ ベクトルDB構築完了")

    def search(self, query: str, top_k: int = 5) -> Tuple[List[str], List[float]]:
        """クエリに基づいて検索"""
        query_embedding = self.embedder.encode([query])[0]

        results = self.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k
        )

        documents = results['documents'][0]
        distances = results['distances'][0]

        # 距離をスコアに変換（距離が小さいほどスコアが高い）
        scores = [1 / (1 + d) for d in distances]

        return documents, scores


    def query(self, question: str) -> Tuple[str, List[str], List[float], float, float, float, float]:
        """質問に回答"""
        total_start_time = time.time()

        # 1. ベクトル検索
        search_start_time = time.time()
        documents, scores = self.search(question, top_k=self.config.top_k) # self.vector_db.search から変更
        search_time = time.time() - search_start_time

        # 2. Reranking（オプション）
        rerank_time = 0.0
        if self.reranker:
            rerank_start_time = time.time()
            documents, scores = self.reranker.rerank(
                question,
                documents,
                top_k=self.config.rerank_top_k
            )
            rerank_time = time.time() - rerank_start_time
        else:
             # Rerankerを使用しない場合、top_kで取得したドキュメントとスコアをそのまま使用
             # ただし、ExperimentResultに格納する際にはrerank_top_kの数に合わせる
             if len(documents) > self.config.rerank_top_k:
                 documents = documents[:self.config.rerank_top_k]
                 scores = scores[:self.config.rerank_top_k]


        # 3. プロンプト構築

        # 4. LLM生成
        llm_start_time = time.time()
        context = "\n\n".join([f"[参考{i+1}]\n{doc}" for i, doc in enumerate(documents)])
        prompt = f"""以下の参考情報を基に、質問に回答してください。

参考情報：
{context}

質問：{question}

回答："""
        answer = self.llm.generate(prompt)
        llm_time = time.time() - llm_start_time

        total_execution_time = time.time() - total_start_time

        return answer, documents, scores, total_execution_time, search_time, rerank_time, llm_time

In [14]:
# ================================================================================
# 実験管理
# ================================================================================

class ExperimentManager:
    """複数設定での性能比較実験"""

    def __init__(self, pdf_path: str):
        self.pdf_path = pdf_path
        self.results: List[ExperimentResult] = []
        self.llm = None # LLMインスタンスを保持する変数
        self.vector_db_client = None # VectorDBクライアントインスタンスを保持する変数
        self.reranker = None # Rerankerインスタンスを保持する変数

    def run_experiments(self, test_queries: List[str]):
        """実験実行"""

        # LLMモデルを一度だけ読み込む
        self.llm = JapaneseLLM()
        # VectorDBクライアントを一度だけ初期化
        self.vector_db_client = chromadb.Client(Settings(anonymized_telemetry=False))
        # Rerankerを一度だけ初期化
        self.reranker = Reranker()

        # 実験設定の組み合わせ
        configs = [
            # 1. Rerankerなし、通常チャンキング、サイズ500、重複10%
            RAGConfig(
                use_reranker=False,
                chunking=ChunkingConfig("recursive", 500, 0, 0.1),
                top_k=5
            ),
            # 2. Rerankerあり、通常チャンキング、サイズ500、重複10%
            RAGConfig(
                use_reranker=True,
                chunking=ChunkingConfig("recursive", 500, 0, 0.1),
                top_k=5,
                rerank_top_k=3
            ),
            # 3. Rerankerあり、サイズ1000、重複20%
            RAGConfig(
                use_reranker=True,
                chunking=ChunkingConfig("recursive", 1000, 0, 0.2),
                top_k=5,
                rerank_top_k=3
            ),
            # 4. Rerankerあり、サイズ300、重複30%
            RAGConfig(
                use_reranker=True,
                chunking=ChunkingConfig("recursive", 300, 0, 0.3),
                top_k=5,
                rerank_top_k=3
            ),
            # 5. Rerankerあり、セマンティックチャンキング、サイズ500、重複10%
            RAGConfig(
                use_reranker=True,
                chunking=ChunkingConfig("semantic", 500, 0, 0.1),
                top_k=5,
                rerank_top_k=3
            ),
        ]

        for i, config in enumerate(configs, 1):
            print(f"\n{'='*70}")
            print(f"実験 {i}/{len(configs)}")
            print(f"Reranker: {config.use_reranker}")
            print(f"チャンキング: {config.chunking.method}")
            print(f"サイズ: {config.chunking.chunk_size}")
            print(f"重複率: {config.chunking.overlap_ratio*100:.0f}%")
            print(f"{'='*70}\n")

            # RAGシステム構築時に既存のインスタンスを渡す
            # VectorDatabaseにはクライアントと新しいコレクション名を渡す
            rag = RAGSystem(
                config=config,
                llm=self.llm,
                chroma_client=self.vector_db_client,
                reranker=self.reranker if config.use_reranker else None,
                collection_name=f"rag_collection_exp_{i}" # 実験ごとに異なるコレクション名を使用
            )
            rag.index_document(self.pdf_path)

            # 各クエリで実験
            for query in test_queries:
                print(f"\n質問: {query}")
                answer, chunks, scores, total_exec_time, search_time, rerank_time, llm_time = rag.query(query)

                result = ExperimentResult(
                    config=config,
                    query=query,
                    answer=answer,
                    execution_time=total_exec_time,
                    retrieved_chunks=chunks,
                    scores=scores,
                    search_time=search_time,
                    rerank_time=rerank_time,
                    llm_time=llm_time
                )
                self.results.append(result)

                print(f"回答: {answer[:100]}...")
                print(f"実行時間: {total_exec_time:.2f}秒 (検索: {search_time:.2f}秒, Rerank: {rerank_time:.2f}秒, LLM: {llm_time:.2f}秒)")

            # 各実験の最後にコレクションを削除してリソースを解放
            try:
                self.vector_db_client.delete_collection(name=f"rag_collection_exp_{i}")
            except:
                pass

            # RAGシステムインスタンスは削除
            del rag
            # LLM, VectorDBクライアント, Rerankerインスタンスは使い回すため削除しない

    def generate_report(self) -> pd.DataFrame:
        """結果レポート生成"""
        data = []
        for r in self.results:
            data.append({
                'Reranker': 'あり' if r.config.use_reranker else 'なし',
                'チャンキング': r.config.chunking.method,
                'チャンクサイズ': r.config.chunking.chunk_size,
                '重複率': f"{r.config.chunking.overlap_ratio*100:.0f}%",
                '質問': r.query[:30] + '...',
                '合計実行時間(秒)': round(r.execution_time, 2),
                '検索時間(秒)': round(r.search_time, 2),
                'Rerank時間(秒)': round(r.rerank_time, 2),
                'LLM時間(秒)': round(r.llm_time, 2),
                '平均スコア': round(np.mean(r.scores), 3)
            })

        df = pd.DataFrame(data)
        return df

    def visualize_results(self):
        """結果の可視化"""
        df = self.generate_report()

        # 実行時間の比較
        fig1 = go.Figure()
        for col in ['検索時間(秒)', 'Rerank時間(秒)', 'LLM時間(秒)']:
             fig1.add_trace(go.Bar(
                name=col,
                x=df.index,
                y=df[col]
            ))
        fig1.update_layout(
            title="各処理時間の比較",
            xaxis_title="実験番号",
            yaxis_title="時間（秒）",
            barmode='group'
        )
        fig1.show()

        # スコアの比較
        fig2 = go.Figure()
        fig2.add_trace(go.Scatter(
            x=df.index,
            y=df['平均スコア'],
            mode='lines+markers',
            name='平均検索スコア'
        ))
        fig2.update_layout(
            title="検索精度の比較",
            xaxis_title="実験番号",
            yaxis_title="平均スコア"
        )
        fig2.show()

In [ ]:
# ================================================================================
# メイン実行
# ================================================================================

def main():
    """メイン処理"""

    # PDFファイルのアップロード
    from google.colab import files
    print("📁 PDFファイルをアップロードしてください")
    uploaded = files.upload()
    pdf_path = list(uploaded.keys())[0]

    # テストクエリ
    test_queries = [
        "この文書の主要なテーマは何ですか？",
        "具体的な数値データを教えてください。",
        "結論や提言は何ですか？"
    ]

    # 実験実行
    manager = ExperimentManager(pdf_path)
    manager.run_experiments(test_queries)

    # 結果表示
    print("\n" + "="*70)
    print("📊 実験結果サマリー")
    print("="*70)
    df = manager.generate_report()
    print(df.to_string(index=False))

    # 可視化
    manager.visualize_results()

    print("\n✅ すべての実験が完了しました")

if __name__ == "__main__":
    main()

📁 PDFファイルをアップロードしてください


Saving 08-0471.pdf to 08-0471 (1).pdf
🤖 LLM読み込み中: elyza/ELYZA-japanese-Llama-2-7b


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✅ LLM読み込み完了
🎯 Rerankerモデル読み込み完了

実験 1/5
Reranker: False
チャンキング: recursive
サイズ: 500
重複率: 10%



[INFO] 2025-10-20 22:14:59,277 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2025-10-20 22:14:59,292 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2025-10-20 22:14:59,292 [RapidOCR] main.py:53: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2025-10-20 22:14:59,378 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2025-10-20 22:14:59,381 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2025-10-20 22:14:59,382 [RapidOCR] main.py:53: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx


🔮 intfloat/multilingual-e5-large 埋め込みモデル読み込み完了
📄 PDF処理開始: 08-0471 (1).pdf


[INFO] 2025-10-20 22:14:59,421 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2025-10-20 22:14:59,453 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_rec_infer.onnx
[INFO] 2025-10-20 22:14:59,453 [RapidOCR] main.py:53: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_rec_infer.onnx
